# PaySim — LightGBM challenger và ensemble với TabICLv2

**Câu hỏi nghiên cứu:** LightGBM đã tune trên validation có vượt TabICLv2 không; nếu không, trung bình có trọng số hai mô hình có cải thiện AP trên giai đoạn tương lai không?

Đây là thí nghiệm tiếp nối, **không ghi đè** kết quả gốc trong `results/paysim_fraud/`; kết quả mới lưu riêng trong `results/paysim_fraud_lgbm/`. PaySim là dữ liệu mô phỏng; trích dẫn và giấy phép CC BY-SA 4.0 như ghi ở Kaggle.

## Thiết kế
- So sánh Logistic Regression, CatBoost, LightGBM và TabICLv2 trên cùng các subset train, validation và test đã chia theo thời gian.
- Tune một lưới nhỏ LightGBM bằng AP trung bình trên validation ở train size 5.000 qua ba seed; dùng cấu hình tốt nhất cho các train size còn lại. Không dùng test để chọn tham số.
- Lưu xác suất validation/test của LightGBM và TabICLv2; chọn một trọng số blend toàn cục dựa trên AP validation trung bình qua các seed/train sizes, rồi báo cáo AP test. Đây là thử nghiệm khám phá trên một validation period, không phải đánh giá xác nhận độc lập.
- So sánh AP/PR-AUC, ROC-AUC; precision/recall/F1 tại ngưỡng được chọn trên validation. Calibration/ngưỡng không làm tăng khả năng xếp hạng AP/ROC-AUC một cách tự thân.

## Chạy trên Kaggle
1. Tạo Kaggle Notebook và thêm PaySim CSV làm **Input dataset**; CSV phải có tên `PS_20174392719_1491204439457_log.csv` hoặc `paysim.csv`.
2. Bật **Internet** trong Notebook options để cài gói còn thiếu và tải checkpoint TabICLv2. Cell đầu tự nhận diện `/kaggle/input/**` và ghi kết quả vào `/kaggle/working/results/paysim_fraud_lgbm/`.
3. Chạy các cell theo thứ tự. TabICLv2 được cấu hình CPU rõ ràng; bật GPU không có nghĩa model này tự dùng GPU.
4. Cuối notebook, tải `paysim_fraud_report_bundle.zip` từ thư mục kết quả. ZIP có CSV metrics/predictions, tuning, provenance, split và README; **không chứa raw dataset**. Trên Kaggle, chọn Save Version với output files để lưu artefact.

## Chạy local
Mở notebook từ project (hoặc thư mục `notebooks/`), bảo đảm CSV ở `datasets/`, rồi chạy các cell theo thứ tự. Kết quả vẫn tách riêng trong `results/paysim_fraud_lgbm/`.

In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata
import importlib.util
import json
import os
import platform
import time

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

IS_KAGGLE = Path('/kaggle/input').is_dir() or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
if IS_KAGGLE:
    ROOT = Path('/kaggle/working')
else:
    ROOT = Path.cwd().resolve()
    if not (ROOT / 'datasets').is_dir():
        if (ROOT.parent / 'datasets').is_dir():
            ROOT = ROOT.parent
        else:
            raise FileNotFoundError('Mở notebook từ thư mục project hoặc notebooks.')

DATASET_CANDIDATES = [
    ROOT / 'datasets' / 'paysim.csv',
    ROOT / 'datasets' / 'PS_20174392719_1491204439457_log.csv',
]
if IS_KAGGLE:
    kaggle_input = Path('/kaggle/input')
    DATASET_CANDIDATES.extend(kaggle_input.glob('**/PS_20174392719_1491204439457_log.csv'))
    DATASET_CANDIDATES.extend(kaggle_input.glob('**/paysim.csv'))
DATASET = next((path for path in DATASET_CANDIDATES if path.is_file()), None)
if DATASET is None:
    raise FileNotFoundError(
        'Không tìm thấy PaySim CSV. Kaggle: Add Input dataset chứa '
        'PS_20174392719_1491204439457_log.csv; local: đặt CSV trong datasets/.'
    )

# Keep the original three-model experiment untouched; this is a separate challenger/ensemble run.
RESULTS = ROOT / 'results' / 'paysim_fraud_lgbm'
RESULTS.mkdir(parents=True, exist_ok=True)
RUNS_FILE = RESULTS / 'runs.csv'
PREDICTIONS_FILE = RESULTS / 'predictions.csv'
SUMMARY_FILE = RESULTS / 'summary.csv'
ENSEMBLE_FILE = RESULTS / 'ensemble_summary.csv'
TUNING_FILE = RESULTS / 'lightgbm_tuning.csv'
CONFIG_FILE = RESULTS / 'config.json'
REPORT_SUMMARY_FILE = RESULTS / 'report_summary.json'
REPORT_README_FILE = RESULTS / 'README_report.md'
REPORT_BUNDLE_FILE = RESULTS / 'paysim_fraud_report_bundle.zip'

# PaySim warns against balance fields; account IDs are used only for history aggregation.
FEATURES = [
    'step',
    'amount',
    'sender_prior_attempt_count',
    'sender_prior_attempted_amount_sum',
    'recipient_prior_attempt_count',
    'recipient_prior_attempted_amount_sum',
]
REQUIRED_COLUMNS = ['step', 'type', 'amount', 'nameOrig', 'nameDest', 'isFraud']
SEEDS = [42, 43, 44]
TRAIN_SIZES = [500, 1000, 2500, 5000]
MAX_EVAL_ROWS = 20_000  # None để đánh giá toàn bộ từng giai đoạn; mặc định là pilot có giới hạn.
PREDICT_BATCH_SIZE = 2_048  # Tự điều chỉnh theo accelerator trong cell kiểm tra môi trường.
EVAL_SAMPLE_SEED = 4242  # Validation/test cố định giữa các seed và mô hình.
CHECKPOINT = 'tabicl-classifier-v2-20260212.ckpt'

PACKAGE_STATUS = {
    name: importlib.util.find_spec(module) is not None
    for name, module in [('catboost', 'catboost'), ('tabicl', 'tabicl'), ('lightgbm', 'lightgbm')]
}


def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return 'not_installed'


VERSIONS = {
    name: package_version(name)
    for name in ['numpy', 'pandas', 'scikit-learn', 'catboost', 'tabicl', 'lightgbm', 'torch']
}
print('Kaggle environment:', IS_KAGGLE)
print('Dataset:', DATASET)
print('Output:', RESULTS)
print('Optional model packages before preflight:', PACKAGE_STATUS)
print('Versions before preflight:', VERSIONS)

Dataset: /Users/thanhpc02/Documents/CaoHoc/Datamining/BaoCao_GiuaKy/Tabular_4SOTA_CPU/datasets/PS_20174392719_1491204439457_log.csv
Output: /Users/thanhpc02/Documents/CaoHoc/Datamining/BaoCao_GiuaKy/Tabular_4SOTA_CPU/results/paysim_fraud_lgbm
Optional model packages: {'catboost': True, 'tabicl': True, 'lightgbm': True}
Versions: {'numpy': '2.5.4', 'pandas': '3.0.6', 'scikit-learn': '1.9.1', 'catboost': '1.2.10', 'tabicl': '2.2.0', 'lightgbm': '4.7.0', 'torch': '2.14.1'}


In [ ]:
# Kaggle: bật Internet để cài các thư viện còn thiếu và tải checkpoint TabICLv2.
# Các gói cài ngay trong phiên notebook; chạy cell này một lần sau cell cấu hình.
import subprocess
import sys

missing_packages = [
    package for package, module in [('catboost', 'catboost'), ('lightgbm', 'lightgbm'), ('tabicl', 'tabicl')]
    if importlib.util.find_spec(module) is None
]
if missing_packages:
    print('Cài gói còn thiếu:', missing_packages, flush=True)
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *missing_packages])
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            'Không cài được dependency. Trên Kaggle bật Internet trong Notebook options rồi chạy lại cell.'
        ) from exc

PACKAGE_STATUS = {
    name: importlib.util.find_spec(module) is not None
    for name, module in [('catboost', 'catboost'), ('tabicl', 'tabicl'), ('lightgbm', 'lightgbm')]
}
VERSIONS = {
    name: package_version(name)
    for name in ['numpy', 'pandas', 'scikit-learn', 'catboost', 'tabicl', 'lightgbm', 'torch']
}
print('Optional model packages:', PACKAGE_STATUS)
print('Runtime versions:', VERSIONS)
if not all(PACKAGE_STATUS.values()):
    raise ImportError(f'Thiếu dependency sau bước cài đặt: {PACKAGE_STATUS}')

## 1. Nạp, kiểm tra và tạo đặc trưng lịch sử

Chỉ đọc `step`, `type`, `amount`, `nameOrig`, `nameDest`, `isFraud`; không nạp các cột số dư hoặc cờ phát hiện. ID tài khoản được dùng tạm thời để tổng hợp số lần/tổng số tiền giao dịch ở các step trước, sau đó bị loại khỏi X. Mọi giao dịch cùng `step` bị loại khỏi lịch sử của nhau vì PaySim không cung cấp thứ tự bên trong giờ. Cell kiểm tra schema, nhãn, giá trị thiếu/hữu hạn rồi lọc `TRANSFER`. Nếu file chưa có, dừng để bạn đặt CSV vào đúng đường dẫn.

In [2]:
import gc

if not DATASET.is_file():
    raise FileNotFoundError(
        f'Chưa có PaySim CSV: {DATASET}. Tải dataset bạn được phép sử dụng và đặt tại đây.'
    )

header = pd.read_csv(DATASET, nrows=0)
missing_columns = sorted(set(REQUIRED_COLUMNS) - set(header.columns))
if missing_columns:
    raise ValueError(f'Thiếu cột PaySim bắt buộc: {missing_columns}')

# Hash theo luồng để không nạp thêm toàn bộ file lớn vào RAM.
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


DATASET_SHA256 = sha256_file(DATASET)
raw = pd.read_csv(
    DATASET,
    usecols=REQUIRED_COLUMNS,
    dtype={
        'step': 'int16',
        'type': 'category',
        'amount': 'float64',
        'nameOrig': 'category',
        'nameDest': 'category',
        'isFraud': 'int8',
    },
)
SOURCE_ROW_COUNT = len(raw)
raw.insert(0, '_source_row', np.arange(SOURCE_ROW_COUNT, dtype=np.int64))
if raw[REQUIRED_COLUMNS].isna().any().any():
    raise ValueError('Có giá trị thiếu trong các cột phân tích; cần xử lý và ghi rõ trước khi chạy.')
if not set(raw['isFraud'].unique()).issubset({0, 1}):
    raise ValueError('isFraud cần được mã hóa 0/1.')
if not np.isfinite(raw[['step', 'amount']].to_numpy(dtype=float)).all():
    raise ValueError('step/amount có giá trị không hữu hạn.')


def make_prior_activity(data, account_column, prefix):
    """Aggregate account activity by hour; remove the current hour from history."""
    per_step = (
        data.groupby([account_column, 'step'], observed=True, sort=False)
        .agg(attempt_count=('amount', 'size'), attempted_amount_sum=('amount', 'sum'))
        .reset_index()
        .sort_values([account_column, 'step'], kind='mergesort')
    )
    account_groups = per_step.groupby(account_column, observed=True, sort=False)
    per_step[f'{prefix}_prior_attempt_count'] = (
        account_groups['attempt_count'].cumsum() - per_step['attempt_count']
    )
    per_step[f'{prefix}_prior_attempted_amount_sum'] = (
        account_groups['attempted_amount_sum'].cumsum() - per_step['attempted_amount_sum']
    )
    return per_step[[
        account_column,
        'step',
        f'{prefix}_prior_attempt_count',
        f'{prefix}_prior_attempted_amount_sum',
    ]]


# History includes past observed attempts of every transaction type, not labels or balances.
# Group by hour and subtract the current hour, since within-hour order is unavailable.
sender_history = make_prior_activity(raw, 'nameOrig', 'sender')
recipient_history = make_prior_activity(raw, 'nameDest', 'recipient')
targets = raw.loc[raw['type'].eq('TRANSFER')].copy()
targets = targets.merge(
    sender_history, on=['nameOrig', 'step'], how='left', validate='many_to_one'
)
targets = targets.merge(
    recipient_history, on=['nameDest', 'step'], how='left', validate='many_to_one'
)

history_features = [
    'sender_prior_attempt_count',
    'sender_prior_attempted_amount_sum',
    'recipient_prior_attempt_count',
    'recipient_prior_attempted_amount_sum',
]
targets[history_features] = targets[history_features].fillna(0)
frame = targets.drop(columns=['type', 'nameOrig', 'nameDest']).reset_index(drop=True)
X = frame[FEATURES].to_numpy(dtype=np.float64)
y = frame['isFraud'].to_numpy(dtype=np.int8)
if frame.empty or np.unique(y).size != 2:
    raise ValueError('Dữ liệu TRANSFER cần có cả giao dịch hợp lệ và gian lận.')
if not np.isfinite(X).all():
    raise ValueError('Đặc trưng sau khi tạo lịch sử có giá trị không hữu hạn.')

print(f'CSV nguồn: {SOURCE_ROW_COUNT:,} dòng | TRANSFER: {len(frame):,} dòng')
print('Nhãn trong TRANSFER:', {int(k): int(v) for k, v in zip(*np.unique(y, return_counts=True))})
print('Prevalence gian lận:', f'{y.mean():.6%}')
print('Đặc trưng:', FEATURES)
print('SHA-256:', DATASET_SHA256)
display(frame[['_source_row', *FEATURES, 'isFraud']].head())

del raw, targets, sender_history, recipient_history
gc.collect()

CSV nguồn: 6,362,620 dòng | TRANSFER: 532,909 dòng
Nhãn trong TRANSFER: {0: 528812, 1: 4097}
Prevalence gian lận: 0.768799%
Đặc trưng: ['step', 'amount', 'sender_prior_attempt_count', 'sender_prior_attempted_amount_sum', 'recipient_prior_attempt_count', 'recipient_prior_attempted_amount_sum']
SHA-256: 16910f90577b0d981bf8ff289714510bb89bc71bff7d3f220f024e287e4eea6b


,_source_row,step,amount,sender_prior_attempt_count,sender_prior_attempted_amount_sum,recipient_prior_attempt_count,recipient_prior_attempted_amount_sum,isFraud
0,2,1,181.00,0,0.0,0,0.0,1
1,19,1,215310.30,0,0.0,0,0.0,0
2,24,1,311685.89,0,0.0,0,0.0,0
3,58,1,62610.80,0,0.0,0,0.0,0
4,78,1,42712.39,0,0.0,0,0.0,0


16

## 2. Chia theo thời gian và tạo mẫu đánh giá cố định

Các ranh giới được xác định từ số `step` phân biệt (60% / 80%), không xáo trộn thời gian. Validation và test được lấy mẫu ngẫu nhiên **không phân tầng** trong giai đoạn tương ứng để gần giữ prevalence tự nhiên; cùng các dòng đánh giá được dùng cho mọi mô hình/seed. Hash và ranh giới được lưu để tái lập.

In [3]:
unique_steps = np.sort(frame['step'].unique())
if len(unique_steps) < 5:
    raise ValueError('Cần tối thiểu 5 giá trị step để tạo chia thời gian có ý nghĩa.')

train_step_count = int(len(unique_steps) * 0.60)
val_step_count = int(len(unique_steps) * 0.80)
train_step_count = min(max(train_step_count, 1), len(unique_steps) - 2)
val_step_count = min(max(val_step_count, train_step_count + 1), len(unique_steps) - 1)
TRAIN_STEP_END = float(unique_steps[train_step_count - 1])
VAL_STEP_END = float(unique_steps[val_step_count - 1])

train_pool_idx = np.flatnonzero(frame['step'].to_numpy() <= TRAIN_STEP_END)
val_period_idx = np.flatnonzero(
    (frame['step'].to_numpy() > TRAIN_STEP_END)
    & (frame['step'].to_numpy() <= VAL_STEP_END)
)
test_period_idx = np.flatnonzero(frame['step'].to_numpy() > VAL_STEP_END)


def sample_eval_indices(indices, seed):
    if MAX_EVAL_ROWS is None or len(indices) <= MAX_EVAL_ROWS:
        return np.asarray(indices, dtype=np.int64)
    rng = np.random.default_rng(seed)
    return np.sort(rng.choice(indices, size=MAX_EVAL_ROWS, replace=False))


val_idx = sample_eval_indices(val_period_idx, EVAL_SAMPLE_SEED)
test_idx = sample_eval_indices(test_period_idx, EVAL_SAMPLE_SEED + 1)
if any(np.unique(y[idx]).size != 2 for idx in [train_pool_idx, val_idx, test_idx]):
    raise ValueError('Train/validation/test cần có cả hai lớp; kiểm tra dữ liệu hoặc khoảng thời gian.')


def hash_indices(indices):
    return hashlib.sha256(np.asarray(indices, dtype=np.int64).tobytes()).hexdigest()


SPLIT_CONFIG = {
    'dataset_sha256': DATASET_SHA256,
    'strategy': 'chronological by unique step; first 60% train, next 20% validation, final 20% test',
    'train_step_end': TRAIN_STEP_END,
    'validation_step_end': VAL_STEP_END,
    'train_pool_rows': int(len(train_pool_idx)),
    'validation_period_rows': int(len(val_period_idx)),
    'test_period_rows': int(len(test_period_idx)),
    'validation_eval_rows': int(len(val_idx)),
    'test_eval_rows': int(len(test_idx)),
    'validation_prevalence': float(y[val_idx].mean()),
    'test_prevalence': float(y[test_idx].mean()),
    'validation_indices_sha256': hash_indices(val_idx),
    'test_indices_sha256': hash_indices(test_idx),
    'max_eval_rows_per_split': MAX_EVAL_ROWS,
    'eval_sample_seed': EVAL_SAMPLE_SEED,
}
(RESULTS / 'split_config.json').write_text(json.dumps(SPLIT_CONFIG, indent=2), encoding='utf-8')

print('Step ranh giới: train <=', TRAIN_STEP_END, '| validation <=', VAL_STEP_END, '| test sau đó')
for name, indices in [('train pool', train_pool_idx), ('validation', val_idx), ('test', test_idx)]:
    print(f'{name:12s}: n={len(indices):,}, fraud={int(y[indices].sum()):,}, prevalence={y[indices].mean():.6%}')
print('Validation/test indices SHA-256:', SPLIT_CONFIG['validation_indices_sha256'][:12], SPLIT_CONFIG['test_indices_sha256'][:12])

Step ranh giới: train <= 445.0 | validation <= 594.0 | test sau đó
train pool  : n=499,633, fraud=2,494, prevalence=0.499166%
validation  : n=20,000, fraud=722, prevalence=3.610000%
test        : n=12,029, fraud=827, prevalence=6.875052%
Validation/test indices SHA-256: 1b25d54c6ef3 314ee37ae0b6


## 3. Mẫu train dùng chung

Lấy mẫu phân tầng độc lập từ giai đoạn lịch sử để các mức train có đủ hai lớp; mỗi seed/kích thước có đúng cùng chỉ số cho cả ba mô hình. Vì việc lấy mẫu phân tầng làm prevalence train khác tự nhiên, không diễn giải xác suất thô như xác suất rủi ro đã hiệu chỉnh.

In [4]:
TRAIN_SUBSETS = {}
for seed in SEEDS:
    TRAIN_SUBSETS[str(seed)] = {}
    for n_train in TRAIN_SIZES:
        if n_train > len(train_pool_idx):
            raise ValueError(f'Train size {n_train} lớn hơn train pool {len(train_pool_idx)}.')
        if n_train < 2:
            raise ValueError('Mỗi train subset cần tối thiểu hai mẫu.')
        chosen, _ = train_test_split(
            train_pool_idx,
            train_size=n_train,
            stratify=y[train_pool_idx],
            random_state=seed,
        )
        if np.unique(y[chosen]).size != 2:
            raise ValueError(f'Subset seed={seed}, n={n_train} không có đủ hai lớp.')
        TRAIN_SUBSETS[str(seed)][str(n_train)] = np.sort(chosen).astype(int).tolist()
        print(
            f'seed={seed}, n={n_train}: fraud={int(y[chosen].sum())}, '
            f'prevalence={y[chosen].mean():.4%}, hash={hash_indices(chosen)[:12]}'
        )

train_subset_payload = {
    'dataset_sha256': DATASET_SHA256,
    'split_config': SPLIT_CONFIG,
    'seeds': SEEDS,
    'train_sizes': TRAIN_SIZES,
    'indices': TRAIN_SUBSETS,
}
(RESULTS / 'train_subsets.json').write_text(
    json.dumps(train_subset_payload), encoding='utf-8'
)
print('Đã lưu mẫu train dùng chung:', RESULTS / 'train_subsets.json')

seed=42, n=500: fraud=2, prevalence=0.4000%, hash=60a20ce30a5c
seed=42, n=1000: fraud=5, prevalence=0.5000%, hash=543e4dc7123d
seed=42, n=2500: fraud=12, prevalence=0.4800%, hash=e44731afd283
seed=42, n=5000: fraud=25, prevalence=0.5000%, hash=4b5f6ddb8a95
seed=43, n=500: fraud=2, prevalence=0.4000%, hash=442c4df6931f
seed=43, n=1000: fraud=5, prevalence=0.5000%, hash=117d8f41cefa
seed=43, n=2500: fraud=12, prevalence=0.4800%, hash=52487dbc70c9
seed=43, n=5000: fraud=25, prevalence=0.5000%, hash=aca3c4a3a020
seed=44, n=500: fraud=2, prevalence=0.4000%, hash=163c789388b2
seed=44, n=1000: fraud=5, prevalence=0.5000%, hash=4ac61dee13a8
seed=44, n=2500: fraud=12, prevalence=0.4800%, hash=ed19c6749a30
seed=44, n=5000: fraud=25, prevalence=0.5000%, hash=797e2067374b
Đã lưu mẫu train dùng chung: /Users/thanhpc02/Documents/CaoHoc/Datamining/BaoCao_GiuaKy/Tabular_4SOTA_CPU/results/paysim_fraud_lgbm/train_subsets.json


## 4. Định nghĩa mô hình và metric

Logistic Regression được scale trong pipeline chỉ fit trên train; CatBoost dùng class weighting cân bằng; TabICLv2 dùng checkpoint v2 cố định. Không tuning mô hình bằng test. Ngưỡng phân loại được chọn bằng F1 trên validation và sau đó khóa trước khi đánh giá test; AP/ROC-AUC là metric xếp hạng không phụ thuộc ngưỡng.

In [ ]:
MODEL_IDS = ['LogisticRegression', 'CatBoost', 'LightGBM', 'TabICLv2']
LOGISTIC_PARAMS = {'class_weight': 'balanced', 'max_iter': 1000, 'solver': 'lbfgs'}
CATBOOST_PARAMS = {
    'iterations': 500,
    'depth': 6,
    'learning_rate': 0.05,
    'loss_function': 'Logloss',
    'auto_class_weights': 'Balanced',
    'verbose': False,
}
LIGHTGBM_PARAMS = {
    'objective': 'binary',
    'n_estimators': 300,
    'learning_rate': 0.05,
    'num_leaves': 15,
    'min_child_samples': 20,
    'class_weight': 'balanced',
    'reg_lambda': 1.0,
    'verbosity': -1,
    'n_jobs': min(4, os.cpu_count() or 2),
    'force_col_wise': True,
}
TABICL_PARAMS = {
    # Keep CPU explicit: Kaggle accelerator availability does not imply TabICL support.
    'device': 'cpu',
    'n_estimators': 1,
    'batch_size': 1,
    'use_amp': False,
    'use_fa3': False,
    'offload_mode': 'auto',
    'checkpoint_version': CHECKPOINT,
    'n_jobs': min(4, os.cpu_count() or 2),
}


def build_model(model_id, seed):
    if model_id == 'LogisticRegression':
        return make_pipeline(StandardScaler(), LogisticRegression(**LOGISTIC_PARAMS, random_state=seed))
    if model_id == 'CatBoost':
        if not PACKAGE_STATUS['catboost']:
            raise ImportError('Thiếu catboost trong kernel đang dùng.')
        from catboost import CatBoostClassifier

        return CatBoostClassifier(**CATBOOST_PARAMS, random_seed=seed, thread_count=min(4, os.cpu_count() or 2))
    if model_id == 'LightGBM':
        if not PACKAGE_STATUS['lightgbm']:
            raise ImportError('Thiếu lightgbm trong kernel đang dùng.')
        from lightgbm import LGBMClassifier

        return LGBMClassifier(**LIGHTGBM_PARAMS, random_state=seed)
    if model_id == 'TabICLv2':
        if not PACKAGE_STATUS['tabicl']:
            raise ImportError('Thiếu tabicl trong kernel đang dùng.')
        from tabicl import TabICLClassifier

        return TabICLClassifier(**TABICL_PARAMS, random_state=seed)
    raise ValueError(f'Mô hình không được hỗ trợ: {model_id}')


def predict_positive_probability(model, x_eval, stage_label=None):
    class_values = np.asarray(model.classes_)
    positive_positions = np.flatnonzero(class_values == 1)
    if len(positive_positions) != 1:
        raise ValueError(f'Không tìm thấy duy nhất lớp dương 1 trong classes_={class_values}.')
    positive_position = int(positive_positions[0])
    pieces = []
    total_rows = len(x_eval)
    batches = range(0, total_rows, PREDICT_BATCH_SIZE)
    total_batches = (total_rows + PREDICT_BATCH_SIZE - 1) // PREDICT_BATCH_SIZE
    prediction_started = time.perf_counter()
    for batch_number, start in enumerate(batches, start=1):
        batch = x_eval[start : start + PREDICT_BATCH_SIZE]
        batch_started = time.perf_counter()
        if stage_label:
            print(
                f'    {stage_label}: batch {batch_number}/{total_batches} bắt đầu '
                f'({len(batch):,} dòng)',
                flush=True,
            )
        probability = np.asarray(model.predict_proba(batch), dtype=float)
        batch_seconds = time.perf_counter() - batch_started
        if probability.ndim != 2 or probability.shape[1] != len(class_values):
            raise ValueError(f'Shape predict_proba không khớp classes_: {probability.shape}.')
        pieces.append(probability[:, positive_position])
        if stage_label:
            print(
                f'    {stage_label}: batch {batch_number}/{total_batches} xong trong '
                f'{batch_seconds:.1f}s ({len(batch) / max(batch_seconds, 1e-9):,.1f} dòng/s); '
                f'đã xử lý {min(start + len(batch), total_rows):,}/{total_rows:,}',
                flush=True,
            )
    result = np.concatenate(pieces)
    total_seconds = time.perf_counter() - prediction_started
    if stage_label:
        print(
            f'    {stage_label}: hoàn tất {total_rows:,} dòng trong {total_seconds:.1f}s '
            f'({total_rows / max(total_seconds, 1e-9):,.1f} dòng/s)',
            flush=True,
        )
    if not np.isfinite(result).all() or ((result < 0) | (result > 1)).any():
        raise ValueError('Xác suất lớp dương không hợp lệ.')
    return result


def threshold_from_validation(y_true, probability):
    precision, recall, thresholds = precision_recall_curve(y_true, probability)
    if len(thresholds) == 0:
        return 0.5, 0.0
    f1_values = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    best = int(np.nanargmax(f1_values))
    return float(thresholds[best]), float(f1_values[best])


RUN_CONFIG = {
    'dataset': 'PaySim synthetic financial transactions',
    'dataset_path': str(DATASET),
    'dataset_sha256': DATASET_SHA256,
    'source_rows': SOURCE_ROW_COUNT,
    'transfer_rows': int(len(frame)),
    'features': FEATURES,
    'excluded_fields': ['isFraud', 'isFlaggedFraud', 'all balance columns', 'account IDs from X'],
    'positive_class': 1,
    'split_config': SPLIT_CONFIG,
    'seeds': SEEDS,
    'train_sizes': TRAIN_SIZES,
    'models': MODEL_IDS,
    'logistic_params': LOGISTIC_PARAMS,
    'catboost_params': CATBOOST_PARAMS,
    'lightgbm_params': LIGHTGBM_PARAMS,
    'tabicl_params': TABICL_PARAMS,
    'package_versions': VERSIONS,
    'python': platform.python_version(),
    'platform': platform.platform(),
    'is_kaggle': IS_KAGGLE,
    'device_note': 'TabICLv2 is explicitly configured for CPU; no GPU assumption is made.',
    'selection_note': 'LightGBM tuning and ensemble weight selected on validation only; test used for final evaluation.',
}
CONFIG_FILE.write_text(json.dumps(RUN_CONFIG, indent=2), encoding='utf-8')
print('Đã lưu cấu hình:', CONFIG_FILE)

Đã lưu cấu hình: /Users/thanhpc02/Documents/CaoHoc/Datamining/BaoCao_GiuaKy/Tabular_4SOTA_CPU/results/paysim_fraud_lgbm/config.json


In [6]:
RUN_CONFIG.update({
    'source_url': 'https://www.kaggle.com/datasets/ealaxi/paysim1',
    'license': 'CC BY-SA 4.0 (as listed on Kaggle dataset page)',
    'citation': 'Lopez-Rojas, E. A., Elmir, A., and Axelsson, S. (2016). PaySim: A financial mobile money simulator for fraud detection. EMSS.',
    'excluded_fields': [
        'isFraud from X', 'isFlaggedFraud', 'oldbalanceOrg', 'newbalanceOrig',
        'oldbalanceDest', 'newbalanceDest', 'nameOrig from X', 'nameDest from X',
    ],
    'history_features': {
        'sender_prior_attempt_count': 'number of prior-step transactions initiated by nameOrig',
        'sender_prior_attempted_amount_sum': 'sum of prior-step amounts initiated by nameOrig',
        'recipient_prior_attempt_count': 'number of prior-step transactions received by nameDest',
        'recipient_prior_attempted_amount_sum': 'sum of prior-step amounts received by nameDest',
    },
    'history_temporal_rule': 'all transaction types; strictly lower step only; all same-step events excluded; no label or balance used',
    'history_data_access': 'Use all transaction logs as unlabeled history; model fit uses only selected labeled training subset.',
    'label_budget_protocol': 'Stratified labeled subset using known labels for controlled sample-size experiment; not random-label acquisition simulation.',
    'data_card_caveat': 'PaySim warns balance columns are unsuitable because detected fraud transactions are cancelled.',
})
CONFIG_FILE.write_text(json.dumps(RUN_CONFIG, indent=2), encoding='utf-8')
print('Đã lưu provenance, leakage exclusions và quy tắc history/label-budget vào config.')

Đã lưu provenance, leakage exclusions và quy tắc history/label-budget vào config.


In [7]:
if not PACKAGE_STATUS['lightgbm']:
    raise ImportError('LightGBM chưa được cài trong kernel.')
from lightgbm import LGBMClassifier

TUNING_CANDIDATES = [
    {'num_leaves': leaves, 'min_child_samples': child_samples}
    for leaves, child_samples in [(7, 10), (7, 30), (15, 10), (15, 30), (31, 10), (31, 30)]
]
tuning_rows = []
TUNE_TRAIN_SIZE = 5000
for candidate_id, candidate in enumerate(TUNING_CANDIDATES):
    candidate_scores = []
    for seed in SEEDS:
        train_idx = np.asarray(TRAIN_SUBSETS[str(seed)][str(TUNE_TRAIN_SIZE)], dtype=np.int64)
        params = {**LIGHTGBM_PARAMS, **candidate, 'random_state': seed}
        model = LGBMClassifier(**params)
        model.fit(X[train_idx], y[train_idx])
        val_probability = predict_positive_probability(model, X[val_idx])
        ap = average_precision_score(y[val_idx], val_probability)
        candidate_scores.append(ap)
        tuning_rows.append({
            'candidate_id': candidate_id,
            'seed': seed,
            'train_size': TUNE_TRAIN_SIZE,
            **candidate,
            'validation_average_precision': ap,
        })
        print(f'candidate={candidate_id}, seed={seed}, validation AP={ap:.5f}')
        del model
    print(f'candidate={candidate_id}, mean validation AP={np.mean(candidate_scores):.5f}')

tuning = pd.DataFrame(tuning_rows)
tuning.to_csv(TUNING_FILE, index=False)
mean_scores = tuning.groupby('candidate_id')['validation_average_precision'].mean()
best_candidate_id = int(mean_scores.idxmax())
best_candidate = TUNING_CANDIDATES[best_candidate_id]
LIGHTGBM_PARAMS.update(best_candidate)
RUN_CONFIG['lightgbm_params'] = LIGHTGBM_PARAMS
RUN_CONFIG['lightgbm_tuning'] = {
    'metric': 'mean validation average precision across seeds',
    'train_size': TUNE_TRAIN_SIZE,
    'candidates': TUNING_CANDIDATES,
    'best_candidate_id': best_candidate_id,
    'best_candidate_mean_validation_ap': float(mean_scores.loc[best_candidate_id]),
}
CONFIG_FILE.write_text(json.dumps(RUN_CONFIG, indent=2), encoding='utf-8')
print('Selected LightGBM params:', LIGHTGBM_PARAMS)
print('Tuning table:', TUNING_FILE)
display(tuning.groupby(['candidate_id', 'num_leaves', 'min_child_samples'])['validation_average_precision'].agg(['mean', 'std']).reset_index())

candidate=0, seed=42, validation AP=0.23810
candidate=0, seed=43, validation AP=0.28211
candidate=0, seed=44, validation AP=0.24083
candidate=0, mean validation AP=0.25368
candidate=1, seed=42, validation AP=0.24070
candidate=1, seed=43, validation AP=0.26716
candidate=1, seed=44, validation AP=0.25541
candidate=1, mean validation AP=0.25442
candidate=2, seed=42, validation AP=0.26281
candidate=2, seed=43, validation AP=0.28227
candidate=2, seed=44, validation AP=0.23375
candidate=2, mean validation AP=0.25961
candidate=3, seed=42, validation AP=0.28527
candidate=3, seed=43, validation AP=0.27696
candidate=3, seed=44, validation AP=0.23972
candidate=3, mean validation AP=0.26732
candidate=4, seed=42, validation AP=0.28836
candidate=4, seed=43, validation AP=0.28389
candidate=4, seed=44, validation AP=0.23919
candidate=4, mean validation AP=0.27048
candidate=5, seed=42, validation AP=0.28992
candidate=5, seed=43, validation AP=0.28451
candidate=5, seed=44, validation AP=0.26019
candidat

,candidate_id,num_leaves,min_child_samples,mean,std
0,0,7,10,0.253682,0.024661
1,1,7,30,0.254420,0.013259
2,2,15,10,0.259610,0.024417
3,3,15,30,0.267319,0.024262
4,4,31,10,0.270478,0.027192
5,5,31,30,0.278207,0.015833


## 5. Chạy thực nghiệm

Cell này chạy 3 seed × 4 train sizes × 4 mô hình (48 lượt). TabICLv2 có thể tốn thời gian đáng kể. Để smoke test, giảm tạm `SEEDS` còn `[42]`, `TRAIN_SIZES` còn `[500, 1000]` và `MAX_EVAL_ROWS` còn `5000`, rồi khôi phục trước khi chạy đầy đủ. Mỗi lượt ghi kết quả; xác suất validation/test của LightGBM và TabICLv2 được lưu riêng để đánh giá ensemble.

In [ ]:
for stale in [RUNS_FILE, PREDICTIONS_FILE, SUMMARY_FILE, ENSEMBLE_FILE]:
    stale.unlink(missing_ok=True)

run_rows = []
y_val = y[val_idx]
y_test = y[test_idx]
ensemble_models = {'LightGBM', 'TabICLv2'}
total_runs = len(SEEDS) * len(TRAIN_SIZES) * len(MODEL_IDS)
run_index = 0
experiment_started = time.perf_counter()

for seed in SEEDS:
    for n_train in TRAIN_SIZES:
        train_idx = np.asarray(TRAIN_SUBSETS[str(seed)][str(n_train)], dtype=np.int64)
        x_train, y_train = X[train_idx], y[train_idx]
        for model_id in MODEL_IDS:
            run_index += 1
            row = {
                'model': model_id,
                'seed': seed,
                'train_size': n_train,
                'train_fraud': int(y_train.sum()),
                'train_prevalence': float(y_train.mean()),
                'validation_size': int(len(val_idx)),
                'validation_fraud': int(y_val.sum()),
                'validation_prevalence': float(y_val.mean()),
                'test_size': int(len(test_idx)),
                'test_fraud': int(y_test.sum()),
                'test_prevalence': float(y_test.mean()),
                'dataset_sha256': DATASET_SHA256,
                'train_indices_sha256': hash_indices(train_idx),
                'validation_indices_sha256': SPLIT_CONFIG['validation_indices_sha256'],
                'test_indices_sha256': SPLIT_CONFIG['test_indices_sha256'],
                'threshold_from_validation': np.nan,
                'validation_f1_at_threshold': np.nan,
                'validation_average_precision': np.nan,
                'test_average_precision': np.nan,
                'test_roc_auc': np.nan,
                'test_precision': np.nan,
                'test_recall': np.nan,
                'test_f1': np.nan,
                'tn': np.nan,
                'fp': np.nan,
                'fn': np.nan,
                'tp': np.nan,
                'fit_seconds': np.nan,
                'validation_predict_seconds': np.nan,
                'test_predict_seconds': np.nan,
                'status': 'failed',
                'error': '',
            }
            run_started = time.perf_counter()
            print(
                f'[{run_index}/{total_runs}] Bắt đầu {model_id}: seed={seed}, train={n_train} '
                f'| tổng thời gian {run_started - experiment_started:.1f}s',
                flush=True,
            )
            try:
                stage_started = time.perf_counter()
                print('  Bước 1/5: khởi tạo model/checkpoint...', flush=True)
                model = build_model(model_id, seed)
                print(f'  Khởi tạo xong trong {time.perf_counter() - stage_started:.1f}s', flush=True)

                started = time.perf_counter()
                print(f'  Bước 2/5: fit trên {len(x_train):,} dòng...', flush=True)
                model.fit(x_train, y_train)
                row['fit_seconds'] = time.perf_counter() - started
                print(
                    f'  Fit xong trong {row["fit_seconds"]:.1f}s '
                    f'({len(x_train) / max(row["fit_seconds"], 1e-9):,.1f} dòng/s)',
                    flush=True,
                )

                started = time.perf_counter()
                print(f'  Bước 3/5: dự đoán validation ({len(val_idx):,} dòng)...', flush=True)
                val_probability = predict_positive_probability(
                    model, X[val_idx], stage_label='validation predict'
                )
                row['validation_predict_seconds'] = time.perf_counter() - started
                threshold, validation_f1 = threshold_from_validation(y_val, val_probability)
                row['threshold_from_validation'] = threshold
                row['validation_f1_at_threshold'] = validation_f1
                row['validation_average_precision'] = average_precision_score(y_val, val_probability)

                started = time.perf_counter()
                print(f'  Bước 4/5: dự đoán test ({len(test_idx):,} dòng)...', flush=True)
                test_probability = predict_positive_probability(
                    model, X[test_idx], stage_label='test predict'
                )
                row['test_predict_seconds'] = time.perf_counter() - started
                prediction = (test_probability >= threshold).astype(np.int8)
                tn, fp, fn, tp = confusion_matrix(y_test, prediction, labels=[0, 1]).ravel()
                row.update({
                    'test_average_precision': average_precision_score(y_test, test_probability),
                    'test_roc_auc': roc_auc_score(y_test, test_probability),
                    'test_precision': precision_score(y_test, prediction, zero_division=0),
                    'test_recall': recall_score(y_test, prediction, zero_division=0),
                    'test_f1': f1_score(y_test, prediction, zero_division=0),
                    'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
                    'status': 'ok',
                })
                if model_id in ensemble_models:
                    print('  Bước 5/5: lưu xác suất validation/test...', flush=True)
                    prediction_records = []
                    for split_name, indices, labels, probabilities in [
                        ('validation', val_idx, y_val, val_probability),
                        ('test', test_idx, y_test, test_probability),
                    ]:
                        prediction_records.extend({
                            'model': model_id,
                            'seed': seed,
                            'train_size': n_train,
                            'split': split_name,
                            'row_index': int(index),
                            'label': int(label),
                            'probability': float(probability),
                            'dataset_sha256': DATASET_SHA256,
                            'train_indices_sha256': row['train_indices_sha256'],
                            'validation_indices_sha256': row['validation_indices_sha256'],
                            'test_indices_sha256': row['test_indices_sha256'],
                        } for index, label, probability in zip(indices, labels, probabilities))
                    pd.DataFrame(prediction_records).to_csv(
                        PREDICTIONS_FILE,
                        mode='a',
                        header=not PREDICTIONS_FILE.exists(),
                        index=False,
                    )
                    print('  Đã lưu xác suất.', flush=True)
                print(
                    f'  Kết quả: AP={row["test_average_precision"]:.5f}; '
                    f'ROC-AUC={row["test_roc_auc"]:.5f}; '
                    f'precision={row["test_precision"]:.4f}; recall={row["test_recall"]:.4f}',
                    flush=True,
                )
                del model
            except Exception as exc:
                row['error'] = repr(exc)[:2000]
                print('  FAILED:', row['error'], flush=True)

            row['run_seconds'] = time.perf_counter() - run_started
            run_rows.append(row)
            pd.DataFrame(run_rows).to_csv(RUNS_FILE, index=False)
            print(
                f'[{run_index}/{total_runs}] Đã ghi trạng thái={row["status"]}; '
                f'lượt này {row["run_seconds"]:.1f}s; '
                f'tổng {time.perf_counter() - experiment_started:.1f}s',
                flush=True,
            )

RUN_CONFIG['finished_utc'] = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
RUN_CONFIG['predictions_file'] = str(PREDICTIONS_FILE.relative_to(ROOT))
CONFIG_FILE.write_text(json.dumps(RUN_CONFIG, indent=2), encoding='utf-8')
runs = pd.DataFrame(run_rows)
print('Đã lưu:', RUNS_FILE, 'và', PREDICTIONS_FILE)
display(runs[['model', 'seed', 'train_size', 'train_fraud', 'status', 'test_average_precision',
              'test_roc_auc', 'test_precision', 'test_recall', 'test_f1', 'error']])

[1/48] Bắt đầu LogisticRegression: seed=42, train=500 | tổng thời gian 0.0s
  Bước 1/5: khởi tạo model/checkpoint...
  Khởi tạo xong trong 0.0s
  Bước 2/5: fit trên 500 dòng...
  Fit xong trong 0.0s (34,507.5 dòng/s)
  Bước 3/5: dự đoán validation (20,000 dòng)...
    validation predict: batch 1/10 bắt đầu (2,048 dòng)
    validation predict: batch 1/10 xong trong 0.0s (2,384,514.6 dòng/s); đã xử lý 2,048/20,000
    validation predict: batch 2/10 bắt đầu (2,048 dòng)
    validation predict: batch 2/10 xong trong 0.0s (2,726,424.4 dòng/s); đã xử lý 4,096/20,000
    validation predict: batch 3/10 bắt đầu (2,048 dòng)
    validation predict: batch 3/10 xong trong 0.0s (3,054,437.0 dòng/s); đã xử lý 6,144/20,000
    validation predict: batch 4/10 bắt đầu (2,048 dòng)
    validation predict: batch 4/10 xong trong 0.0s (3,707,908.1 dòng/s); đã xử lý 8,192/20,000
    validation predict: batch 5/10 bắt đầu (2,048 dòng)
    validation predict: batch 5/10 xong trong 0.0s (4,075,281.3 dòng/s); đã

/Users/thanhpc02/Documents/CaoHoc/Datamining/BaoCao_GiuaKy/Tabular_4SOTA_CPU/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 6. Tổng hợp kết quả đã lưu

Chỉ ghép các seed/kích thước có đủ cả ba mô hình thành công và xác nhận chúng dùng cùng tập train, validation, test. Với một seed, độ lệch chuẩn sẽ không có ý nghĩa; cần nhiều seed trước khi đưa ra kết luận.

In [ ]:
runs = pd.read_csv(RUNS_FILE)
if runs.duplicated(['model', 'seed', 'train_size']).any():
    raise ValueError('CSV kết quả có lượt chạy trùng.')

print('Trạng thái lượt chạy:')
display(runs.groupby(['model', 'status']).size().rename('n_runs').reset_index())
successful = runs.loc[runs['status'].eq('ok')].copy()
paired = successful.groupby(['seed', 'train_size']).filter(
    lambda group: set(group['model']) == set(MODEL_IDS)
)
for _, group in paired.groupby(['seed', 'train_size']):
    for key in ['train_indices_sha256', 'validation_indices_sha256', 'test_indices_sha256']:
        if group[key].nunique() != 1:
            raise ValueError(f'Cặp mô hình không dùng cùng {key}.')

METRICS = [
    'validation_average_precision',
    'test_average_precision',
    'test_roc_auc',
    'test_precision',
    'test_recall',
    'test_f1',
]
if paired.empty:
    print('Chưa có nhóm đủ cả bốn mô hình thành công; không tổng hợp kết quả.')
    display(runs.loc[runs['status'].ne('ok'), ['model', 'seed', 'train_size', 'error']])
else:
    summary = paired.groupby(['model', 'train_size'])[METRICS].agg(['mean', 'std', 'count'])
    summary.columns = ['_'.join(column) for column in summary.columns]
    summary = summary.reset_index()
    summary.to_csv(SUMMARY_FILE, index=False)
    print('Kết quả mô hình đơn:')
    display(paired[['model', 'seed', 'train_size', 'train_fraud', 'validation_fraud', 'test_fraud',
                    *METRICS, 'tn', 'fp', 'fn', 'tp']].sort_values(['train_size', 'seed', 'model']))
    print('Mean/std/count trên các seed:')
    display(summary.round(6))

    predictions = pd.read_csv(PREDICTIONS_FILE)
    if predictions.duplicated(['model', 'seed', 'train_size', 'split', 'row_index']).any():
        raise ValueError('File dự đoán có hàng lặp; dừng ensemble.')
    predictions = predictions.loc[predictions['model'].isin(['LightGBM', 'TabICLv2'])].copy()
    blend_pairs = [(int(seed), int(n_train)) for seed, n_train in paired.groupby(['seed', 'train_size']).groups]

    def aligned_pair(seed, n_train, split_name):
        selected = predictions.loc[
            predictions['seed'].eq(seed)
            & predictions['train_size'].eq(n_train)
            & predictions['split'].eq(split_name)
        ]
        scores = selected.pivot(index='row_index', columns='model', values='probability').sort_index()
        labels = selected[['row_index', 'label']].drop_duplicates().set_index('row_index')['label'].sort_index()
        if set(scores.columns) != {'LightGBM', 'TabICLv2'}:
            raise ValueError(f'Thiếu cặp LightGBM/TabICLv2: seed={seed}, n={n_train}, split={split_name}.')
        if not scores.index.equals(labels.index):
            raise ValueError('Những hàng/nhãn của hai mô hình không thẳng hàng.')
        return labels.to_numpy(dtype=np.int8), scores['LightGBM'].to_numpy(), scores['TabICLv2'].to_numpy()

    alphas = np.linspace(0.0, 1.0, 21)  # alpha = weight assigned to TabICLv2
    validation_blend_scores = []
    for alpha in alphas:
        per_pair_ap = []
        for seed, n_train in blend_pairs:
            labels, lgbm_probability, tabicl_probability = aligned_pair(seed, n_train, 'validation')
            blended = (1 - alpha) * lgbm_probability + alpha * tabicl_probability
            per_pair_ap.append(average_precision_score(labels, blended))
        validation_blend_scores.append(float(np.mean(per_pair_ap)))
    best_position = int(np.argmax(validation_blend_scores))
    best_alpha = float(alphas[best_position])
    print(f'Validation chọn alpha TabICLv2={best_alpha:.2f}; mean validation AP={validation_blend_scores[best_position]:.5f}')

    ensemble_rows = []
    for seed, n_train in blend_pairs:
        val_labels, val_lgbm, val_tabicl = aligned_pair(seed, n_train, 'validation')
        test_labels, test_lgbm, test_tabicl = aligned_pair(seed, n_train, 'test')
        val_blended = (1 - best_alpha) * val_lgbm + best_alpha * val_tabicl
        test_blended = (1 - best_alpha) * test_lgbm + best_alpha * test_tabicl
        threshold, validation_f1 = threshold_from_validation(val_labels, val_blended)
        prediction = (test_blended >= threshold).astype(np.int8)
        tn, fp, fn, tp = confusion_matrix(test_labels, prediction, labels=[0, 1]).ravel()
        ensemble_rows.append({
            'model': 'LightGBM+TabICLv2',
            'seed': seed,
            'train_size': n_train,
            'tabicl_weight': best_alpha,
            'validation_average_precision': average_precision_score(val_labels, val_blended),
            'validation_f1_at_threshold': validation_f1,
            'threshold_from_validation': threshold,
            'test_average_precision': average_precision_score(test_labels, test_blended),
            'test_roc_auc': roc_auc_score(test_labels, test_blended),
            'test_precision': precision_score(test_labels, prediction, zero_division=0),
            'test_recall': recall_score(test_labels, prediction, zero_division=0),
            'test_f1': f1_score(test_labels, prediction, zero_division=0),
            'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
        })
    ensemble = pd.DataFrame(ensemble_rows)
    ensemble.to_csv(ENSEMBLE_FILE, index=False)
    ensemble_summary = ensemble.groupby('train_size')[METRICS].agg(['mean', 'std', 'count'])
    ensemble_summary.columns = ['_'.join(column) for column in ensemble_summary.columns]
    ensemble_summary = ensemble_summary.reset_index()
    ensemble_summary.to_csv(RESULTS / 'ensemble_metrics_summary.csv', index=False)
    print('Kết quả ensemble (alpha đã chọn trên validation; test chưa dùng để chọn):')
    display(ensemble.round(6))
    display(ensemble_summary.round(6))
    print('Bảng mô hình:', SUMMARY_FILE)
    print('Chi tiết ensemble:', ENSEMBLE_FILE)
    print('Dự đoán đã lưu:', PREDICTIONS_FILE)

Trạng thái lượt chạy:


,model,status,n_runs
0,CatBoost,ok,12
1,LogisticRegression,ok,12
2,TabICLv2,ok,12


Kết quả paired:


,model,seed,train_size,train_fraud,validation_fraud,test_fraud,validation_average_precision,test_average_precision,test_roc_auc,test_precision,test_recall,test_f1,tn,fp,fn,tp
1,CatBoost,42,500,2,722,827,0.191435,0.360548,0.901713,0.550296,0.337364,0.418291,10974,228,548,279
0,LogisticRegression,42,500,2,722,827,0.229361,0.454853,0.907734,0.179159,1.000000,0.303877,7413,3789,0,827
2,TabICLv2,42,500,2,722,827,0.232998,0.464484,0.948391,0.345015,1.000000,0.513027,9632,1570,0,827
13,CatBoost,43,500,2,722,827,0.176767,0.272807,0.890604,0.274940,0.696493,0.394251,9683,1519,251,576
12,LogisticRegression,43,500,2,722,827,0.215110,0.226153,0.767129,0.100000,0.001209,0.002389,11193,9,826,1
14,TabICLv2,43,500,2,722,827,0.169006,0.313158,0.898856,0.188083,1.000000,0.316616,7632,3570,0,827
25,CatBoost,44,500,2,722,827,0.225517,0.316103,0.919563,0.343302,0.985490,0.509216,9643,1559,12,815
24,LogisticRegression,44,500,2,722,827,0.141834,0.317177,0.794351,0.155838,0.785973,0.260104,7681,3521,177,650
26,TabICLv2,44,500,2,722,827,0.250546,0.322342,0.914228,0.319527,0.783555,0.453940,9822,1380,179,648
4,CatBoost,42,1000,5,722,827,0.275742,0.436380,0.894845,0.554000,0.334946,0.417483,10979,223,550,277


Mean/std/count trên các seed (std không diễn giải được nếu chỉ có một seed):


,model,train_size,validation_average_precision_mean,validation_average_precision_std,validation_average_precision_count,test_average_precision_mean,test_average_precision_std,test_average_precision_count,test_roc_auc_mean,test_roc_auc_std,test_roc_auc_count,test_precision_mean,test_precision_std,test_precision_count,test_recall_mean,test_recall_std,test_recall_count,test_f1_mean,test_f1_std,test_f1_count
0,CatBoost,500,0.197906,0.025011,3,0.316486,0.043872,3,0.903960,0.014610,3,0.389513,0.143376,3,0.673116,0.324695,3,0.440586,0.060639,3
1,CatBoost,1000,0.259353,0.019122,3,0.414958,0.071581,3,0.922901,0.025607,3,0.430704,0.107164,3,0.690447,0.331660,3,0.482431,0.057792,3
2,CatBoost,2500,0.271601,0.036113,3,0.430893,0.069941,3,0.926263,0.024747,3,0.386830,0.028464,3,0.786780,0.240554,3,0.506733,0.036181,3
3,CatBoost,5000,0.284405,0.044704,3,0.462549,0.043448,3,0.944040,0.005468,3,0.367265,0.003451,3,0.938331,0.094526,3,0.527008,0.012319,3
4,LogisticRegression,500,0.195435,0.046964,3,0.332728,0.115140,3,0.823072,0.074572,3,0.144999,0.040678,3,0.595728,0.525872,3,0.188790,0.162905,3
5,LogisticRegression,1000,0.230832,0.039282,3,0.377429,0.027296,3,0.911596,0.011416,3,0.259748,0.094592,3,0.928255,0.068576,3,0.398223,0.114200,3
6,LogisticRegression,2500,0.268950,0.045298,3,0.390162,0.076018,3,0.924428,0.017203,3,0.310296,0.041038,3,0.920193,0.079807,3,0.462488,0.045107,3
7,LogisticRegression,5000,0.296127,0.068929,3,0.472328,0.048507,3,0.939600,0.008470,3,0.368376,0.010189,3,0.876260,0.214325,3,0.513177,0.038919,3
8,TabICLv2,500,0.217517,0.042918,3,0.366661,0.084841,3,0.920491,0.025354,3,0.284208,0.084217,3,0.927852,0.124965,3,0.427861,0.100769,3
9,TabICLv2,1000,0.322896,0.032613,3,0.457566,0.106003,3,0.942612,0.012903,3,0.359908,0.022437,3,1.000000,0.000000,3,0.529043,0.024502,3


Đã lưu: /Users/thanhpc02/Documents/CaoHoc/Datamining/BaoCao_GiuaKy/Tabular_4SOTA_CPU/results/paysim_fraud/summary.csv


In [ ]:
# Reporting package: excludes the raw PaySim CSV; all included tables retain provenance.
import zipfile

artifact_names = [
    'runs.csv', 'predictions.csv', 'summary.csv', 'ensemble_summary.csv',
    'ensemble_metrics_summary.csv', 'lightgbm_tuning.csv', 'config.json',
    'split_config.json', 'train_subsets.json',
]
artifact_paths = [RESULTS / name for name in artifact_names if (RESULTS / name).is_file()]
run_records = pd.read_csv(RUNS_FILE) if RUNS_FILE.is_file() else pd.DataFrame()
completed = int(run_records['status'].eq('ok').sum()) if not run_records.empty else 0
failed = int(run_records['status'].ne('ok').sum()) if not run_records.empty else 0
report_payload = {
    'experiment': 'PaySim LightGBM challenger and LightGBM+TabICLv2 validation-selected blend',
    'generated_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'is_kaggle': IS_KAGGLE,
    'dataset_sha256': DATASET_SHA256,
    'dataset_path_at_runtime': str(DATASET),
    'completed_runs': completed,
    'failed_runs': failed,
    'expected_runs': len(SEEDS) * len(TRAIN_SIZES) * len(MODEL_IDS),
    'included_artifacts': [path.name for path in artifact_paths],
    'raw_dataset_included': False,
    'interpretation_caveats': [
        'PaySim is simulated data, not real-world banking evidence.',
        'Evaluation uses chronological step splits and sampled validation/test rows when configured.',
        'LightGBM configuration and ensemble weight are selected on validation; test is reserved for final evaluation.',
        'TabICLv2 is configured for CPU; Kaggle GPU availability does not imply it was used.',
        'If completed_runs is below expected_runs, the experiment bundle is partial and must be rerun/resumed before final claims.',
    ],
}
if SUMMARY_FILE.is_file():
    report_payload['single_model_summary'] = json.loads(pd.read_csv(SUMMARY_FILE).to_json(orient='records'))
if ENSEMBLE_FILE.is_file():
    report_payload['ensemble_summary'] = json.loads(pd.read_csv(RESULTS / 'ensemble_metrics_summary.csv').to_json(orient='records'))
REPORT_SUMMARY_FILE.write_text(json.dumps(report_payload, indent=2, allow_nan=False), encoding='utf-8')

readme = f'''# PaySim experiment report bundle

Generated: {report_payload['generated_utc']}  
Runtime: {'Kaggle' if IS_KAGGLE else 'local'}  
Dataset SHA-256: `{DATASET_SHA256}`  
Runs: {completed} successful, {failed} failed, {report_payload['expected_runs']} expected.

## Files

- `runs.csv`: per-run status, timing, validation/test metrics and split hashes.
- `predictions.csv`: validation/test probabilities for LightGBM and TabICLv2, with row indices and provenance hashes.
- `summary.csv`: paired single-model metric summary, when at least one complete model group exists.
- `ensemble_summary.csv`: per-seed/train-size blend metrics; `ensemble_metrics_summary.csv` aggregates these by train size.
- `lightgbm_tuning.csv`: validation scores for candidate LightGBM settings.
- `config.json`, `split_config.json`, `train_subsets.json`: protocol, runtime, temporal split, and exact sampled indices.
- `report_summary.json`: concise machine-readable report facts and caveats.
- `manifest.json`: SHA-256 and size for each bundled file, except the manifest itself.

Raw PaySim data is intentionally excluded. The dataset is simulated; do not present results as evidence of real-world bank fraud performance. Check `completed_runs` against `expected_runs` before using results as a complete experiment. The notebook uses CPU for TabICLv2 irrespective of Kaggle accelerator selection.
'''
REPORT_README_FILE.write_text(readme, encoding='utf-8')

manifest = {'bundle': REPORT_BUNDLE_FILE.name, 'created_utc': report_payload['generated_utc'], 'files': []}
for path in [*artifact_paths, REPORT_SUMMARY_FILE, REPORT_README_FILE]:
    manifest['files'].append({
        'path': path.name,
        'bytes': path.stat().st_size,
        'sha256': sha256_file(path),
    })
MANIFEST_FILE = RESULTS / 'manifest.json'
MANIFEST_FILE.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
with zipfile.ZipFile(REPORT_BUNDLE_FILE, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as bundle:
    for path in [*artifact_paths, REPORT_SUMMARY_FILE, REPORT_README_FILE, MANIFEST_FILE]:
        bundle.write(path, arcname=path.name)
print(f'Bundle báo cáo: {REPORT_BUNDLE_FILE} ({REPORT_BUNDLE_FILE.stat().st_size / (1024 ** 2):.1f} MiB)')
print('Trạng thái:', f'{completed}/{report_payload["expected_runs"]} thành công, {failed} thất bại')
print('Chỉ chứa artefact; không đưa CSV PaySim gốc vào ZIP.')